# HCP Aging Data

---

### package imports and basic functions

---

In [1]:
import os
import gc
import sys
import glob
import json
import random
import datetime
import importlib
import itertools
import numpy as np
from scipy import spatial
import scipy.sparse as sparse
import scipy.stats as stats
import pandas as pd
import nibabel as nib
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
import boto3
from tqdm.auto import tqdm


In [2]:
%load_ext autoreload
%autoreload 2

# Path to add to src folder (to use local version of spectranorm)
sys.path.append(os.path.abspath("/mountpoint/code/projects/spectranorm/package/spectranorm/src/"))

from spectranorm import snm


In [3]:
class MyNumpyEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, np.integer):
            return int(obj)
        elif isinstance(obj, np.floating):
            return float(obj)
        elif isinstance(obj, np.ndarray):
            return obj.tolist()
        else:
            return super(MyEncoder, self).default(obj)


def ensure_dir(file_name):
    os.makedirs(os.path.dirname(file_name), exist_ok=True)
    return file_name


def list_dirs(path=os.getcwd()):
    files = glob.glob(os.path.join(path, '*'))
    files = [x for x in files if os.path.isdir(x)]
    return files


def file_exists(file_name, path_name=os.getcwd()):
    return os.path.isfile(os.path.join(path_name, file_name))


def write_json(json_obj, file_path):
    with open(file_path, 'w') as outfile:
        json.dump(json_obj, outfile, sort_keys=True, indent=4,
                  cls=MyNumpyEncoder)
    return json_obj


def load_json(file_path):
    with open(file_path, 'r') as infile:
        return json.load(infile)


def write_np(np_obj, file_path):
    with open(file_path, 'wb') as outfile:
        np.save(outfile, np_obj)


In [4]:
# path setting
main_dir = os.path.abspath('../../..')


## Downloading data from NDA s3 bucket storage

---

In [ ]:
# downloaded by:
# downloadcmd -dp 1205192 -u 'ndehestanikolag' -d AllHCPdataAgingDevelop -wt 8


In [ ]:
# %%bash

# # get list of s3 links to be downloaded
# cat /mountpoint/data/NDA/AllHCPdataAgingDevelop/datastructure_manifest.txt \
# | grep PreprocStrucRecommended \
# | cut -f6 \
# | sed 's/"//g' \
# | grep MNINonLinear/fsaverage_LR32k \
# | grep thickness_MSMAll.32k_fs_LR.dscalar.nii \
# | grep HCA \
# > /mountpoint/data/NDA/HCPD_thickness_s3links.txt
# # | head


In [ ]:
# %%bash

# # make folder to store data in
# mkdir /mountpoint/data/HCP_Aging


In [ ]:
# %%bash

# Download the related files:
# cd /mountpoint/data/
# downloadcmd -dp 1205202 -t /home/sina/Documents/Research/Codes/NDA/HCPA_thickness_s3links.txt  -u 'ndehestanikolag' -d HCP_Aging -wt 8

# other detail in: /mountpoint/code/environments/venv_3.8.10/lib/python3.8/site-packages/NDATools/clientscripts/config/settings.cfg


In [6]:
%%bash

# get list of s3 links to be downloaded
# using freesurfer processed files instead

cat /mountpoint/data/NDA/AllHCPdataAgingDevelop/datastructure_manifest.txt \
| grep PreprocStrucFreesurfer \
| grep \
-e 'lh.white"' -e 'lh.pial"' -e 'lh.thickness"' -e 'lh.orig.nofix"' -e 'lh.sphere.reg"' \
-e 'rh.white"' -e 'rh.pial"' -e 'rh.thickness"' -e 'rh.orig.nofix"' -e 'rh.sphere.reg"' \
| cut -f6 \
| sed 's/"//g' \
> /mountpoint/data/NDA/HCPA_freesurfer_thickness_s3links.txt
# | head -n 12


In [ ]:
# %%bash

# Download the related files:
# cd /mountpoint/data/
# downloadcmd -dp 1184998 -t /mountpoint/data/NDA/HCPA_freesurfer_thickness_s3links.txt -u 'sinamansourlakouraj' -d HCP_Aging -wt 8

# other detail in: /mountpoint/code/environments/venv_3.8.10/lib/python3.8/site-packages/NDATools/clientscripts/config/settings.cfg


## Extracting data

---

In [7]:
hcpa_dir = '/mountpoint/data/HCP_Aging/fmriresults01'
hcpa_subjects = [x.split('/')[-1] for x in list_dirs(hcpa_dir)]
len(hcpa_subjects)


725

In [5]:
hcpa_valid_subjects = [
    subject for subject in hcpa_subjects
    if file_exists(
        f'{hcpa_dir}/{subject}/MNINonLinear/fsaverage_LR32k/{subject}.thickness_MSMAll.32k_fs_LR.dscalar.nii',''
    )
]
len(hcpa_valid_subjects)


725

In [8]:
items = [
    "lh.white", "rh.white",
    "lh.pial", "rh.pial",
    "lh.thickness", "rh.thickness",
    "lh.orig.nofix", "rh.orig.nofix",
    "lh.sphere.reg", "rh.sphere.reg",
]
hcpa_valid_subjects = [
    subject for subject in hcpa_subjects
    if all([
        file_exists(f'/mountpoint/data/HCP_Aging/fmriresults01/{subject}/T1w/{subject}/surf/{item}','')
        for item in items
    ])
]
len(hcpa_valid_subjects)


725

In [9]:
# ignore warning
nib.imageglobals.logger.setLevel(40)


In [10]:
# Store high-resolution thickness for each individual in a separate file
for idx, subject in enumerate(tqdm(hcpa_valid_subjects)):
    sub_dir = f"{idx:02d}"[-2:]
    freesurfer_directory = f"/mountpoint/data/HCP_Aging/fmriresults01/{subject}/T1w/{subject}/"
    transformed_fslr_thickness = snm.utils.nitools.compute_fslr_thickness(freesurfer_directory)
    np.save(
        ensure_dir(f"/mountpoint/data/normative/fs_LR_32k/HCP-A/{sub_dir}/{subject}.thickness.fslr.npy"),
        transformed_fslr_thickness.astype(np.float32)
    )


  0%|          | 0/725 [00:00<?, ?it/s]

In [12]:
hcpa_demography = pd.read_csv(
    f'/mountpoint/data/NDA/AllHCPdataAgingDevelop/fmriresults01.txt',
    delimiter='\t',
    skiprows=[1],
    header=0
)

In [37]:
hcpa_info = pd.read_csv(
    f'/mountpoint/data/NDA/AllHCPdataAgingDevelop/ndar_subject01.txt',
    delimiter='\t',
    skiprows=[1],
    header=0
)


In [13]:
hcpa_ages = np.array(
    [
        float(hcpa_demography[hcpa_demography['src_subject_id'] == (subject[:-6])]['interview_age'].values[0])/12
        for subject in hcpa_valid_subjects
    ]
)
hcpa_ages[:3]

array([39.33333333, 36.5       , 47.33333333])

In [38]:
hcpa_sites = np.array(
    [
        f"site:{hcpa_info.loc[hcpa_info['src_subject_id'] == subject[:-6], 'site'].iloc[0]}"
        for subject in hcpa_valid_subjects
    ]
)
hcpa_sites[:3]

array(['site:MGH', 'site:UMinn', 'site:MGH'], dtype='<U10')

In [14]:
gender_dict = {'M': 0, 'F': 1}
hcpa_genders = np.array(
    [
        hcpa_demography[hcpa_demography['src_subject_id'] == (subject[:-6])]['sex'].values[0]
        for subject in hcpa_valid_subjects
    ]
)
hcpa_genders[:3]

array(['M', 'F', 'M'], dtype='<U1')

In [15]:
hcpa_ids = np.array(
    [
        str(hcpa_demography[hcpa_demography['src_subject_id'] == (subject[:-6])]['src_subject_id'].values[0])
        for subject in hcpa_valid_subjects
    ]
)
hcpa_ids[:3]

array(['HCA6162662', 'HCA8751792', 'HCA8435176'], dtype='<U10')

In [16]:
hcpa_enos = np.array(
    [
        float(snm.utils.nitools.compute_total_euler_number(
            f"/mountpoint/data/HCP_Aging/fmriresults01/{subject}/T1w/{subject}/"
        )) for subject in tqdm(hcpa_valid_subjects)
    ]
)
hcpa_enos[:3]

  0%|          | 0/725 [00:00<?, ?it/s]

array([-36., -30., -54.])

In [17]:
hcpa_mean_thickness = np.array(
    [
        np.load(f"/mountpoint/data/normative/fs_LR_32k/HCP-A/{(idx%100):02d}/{subject}.thickness.fslr.npy").mean()
        for idx, subject in enumerate(tqdm(hcpa_valid_subjects))
    ]
)
hcpa_mean_thickness[:3]

  0%|          | 0/725 [00:00<?, ?it/s]

array([2.6996808, 2.7562442, 2.6877646], dtype=float32)

## Storing cleaned data

---

In [39]:
# mean thickness stored as csv
# hcpa_mean_thickness = np.mean(hcpa_data, axis=1)
hcpa_df = pd.DataFrame({'age': hcpa_ages, 'thickness': hcpa_mean_thickness, 'site': hcpa_sites, 'sex': hcpa_genders, 'subject_ID': hcpa_ids, 'euler_no': hcpa_enos, 'subject_folder': hcpa_valid_subjects, 'subject_index':np.arange(len(hcpa_valid_subjects))})
dataset_name = 'HCP-A'
hcpa_df['dataset'] = dataset_name
hcpa_df[hcpa_df.age < 90].to_parquet(ensure_dir(f'/mountpoint/data/normative/datasets/{dataset_name}/demography.parquet'))


In [36]:
np.save(
    ensure_dir("/mountpoint/data/normative/datasets/HCP-A/subjects.npy"),
    # np.array(hcpa_valid_subjects)[hcpa_df.age < 90]
    np.array(hcpa_valid_subjects)
)


In [13]:
# valid subject names as json
write_json(hcpa_valid_subjects, ensure_dir(f'{main_dir}/data/json/valid_subjects_{dataset_name}.json'));


In [14]:
# high-resolution thickness as npy
write_np(hcpa_data, ensure_dir(f'{main_dir}/data/npy/thickness_{dataset_name}.npy'));


In [40]:
hcpa_df[hcpa_df.age < 90].head()


,age,thickness,site,sex,subject_ID,euler_no,subject_folder,subject_index,dataset
0,39.333333,2.699681,site:MGH,M,HCA6162662,-36.0,HCA6162662_V1_MR,0,HCP-A
1,36.500000,2.756244,site:UMinn,F,HCA8751792,-30.0,HCA8751792_V1_MR,1,HCP-A
2,47.333333,2.687765,site:MGH,M,HCA8435176,-54.0,HCA8435176_V1_MR,2,HCP-A
3,77.666667,2.410398,site:WashU,M,HCA7256575,-78.0,HCA7256575_V1_MR,3,HCP-A
4,57.416667,2.651964,site:UMinn,M,HCA6732374,-54.0,HCA6732374_V1_MR,4,HCP-A


In [31]:
hcpa_df[hcpa_df.age < 90].shape, np.array(hcpa_valid_subjects)[hcpa_df.age < 90].shape

((712, 6), (712,))